# 📚 MODULE 27 — CNN TRAINING & DATA PIPELINE

## 🔥 Complete Revision Notes

> **Scope:** Only the supplied Module 27 block.
>
> **Core idea:** Before worrying about sophisticated CNNs or transfer learning, make sure the **data pipeline is correct**.

---

# 1. 🧠 The Big Picture

A CNN does not simply receive:

```text
Image → CNN
```

The real pipeline is:

```text
                    RAW IMAGES
                        │
                        ▼
                Dataset organization
                        │
                        ▼
             Train / Validation / Test
                        │
              ┌─────────┴─────────┐
              ▼                   ▼
           TRAIN              VAL / TEST
              │                   │
       augmentation               │
              │                   │
              └─────────┬─────────┘
                        ▼
                    preprocessing
                        │
                        ▼
                  Tensor conversion
                        │
                        ▼
                    Normalize
                        │
                        ▼
                    DataLoader
                        │
                        ▼
                       CNN
                        │
                        ▼
                      Loss
                        │
                        ▼
                  Backpropagation
```

PyTorch's official tutorials use this same general pattern with datasets, transforms, `ImageFolder`, `DataLoader`, normalization and augmentation. ([PyTorch Docs][1])

---

# 2. 📦 Dataset

A supervised image-classification dataset consists of pairs:

$$
\boxed{(x_i,y_i)}
$$

where:

* \(x_i\) = image
* \(y_i\) = corresponding class label

Example:

```text
cat001.jpg → cat
cat002.jpg → cat
dog001.jpg → dog
dog002.jpg → dog
```

Mathematically:

$$
D=\{(x_1,y_1),(x_2,y_2),...,(x_N,y_N)\}
$$

The model learns a function:

$$
f_\theta(x)\rightarrow y
$$

where \(\theta\) represents the learnable parameters.

---

# 3. 🗂️ Train / Validation / Test

Split the dataset into separate roles:

```text
                       DATASET
                          │
          ┌───────────────┼───────────────┐
          ▼               ▼               ▼
        TRAIN           VALIDATION       TEST
          │               │               │
      Learn weights    Development      Final
                       decisions       evaluation
```

## 🟢 Train

Used to update model parameters.

```text
Forward
   ↓
Loss
   ↓
Backward
   ↓
Optimizer
   ↓
Weights updated
```

---

## 🟡 Validation

Used during development to make decisions such as:

* learning rate
* augmentation
* architecture
* number of epochs
* regularization
* checkpoint selection

But:

> **Validation data does not directly update the model's weights.**

---

## 🔴 Test

Used for final evaluation.

Ideally:

> The test set should remain unseen throughout model development.

If you repeatedly change the model based on test performance, the test set is effectively becoming part of your development process.

---

# 4. 📊 Example Split

Suppose:

$$
N=10,000
$$

A possible split:

```text
Train       = 8,000
Validation  = 1,000
Test        = 1,000
```

or:

$$
80\%/10\%/10\%
$$

But these are **not universal rules**.

The appropriate split depends on:

* dataset size
* task
* class distribution
* subject identity
* temporal structure
* deployment scenario

---

# 5. 📁 Folder Structure

For a standard classification dataset:

```text
dataset/
│
├── train/
│   ├── cat/
│   │   ├── cat001.jpg
│   │   ├── cat002.jpg
│   │   └── ...
│   │
│   └── dog/
│       ├── dog001.jpg
│       ├── dog002.jpg
│       └── ...
│
├── val/
│   ├── cat/
│   └── dog/
│
└── test/
    ├── cat/
    └── dog/
```

The key convention is:

$$
\boxed{\text{folder name}=\text{class}}
$$

PyTorch's `ImageFolder` expects images organized under class-specific directories such as `root/cat/...` and `root/dog/...`. ([PyTorch Docs][2])

![Image](https://images.openai.com/static-rsc-4/j7rfDLBZoltaG2Zon6-5jYmi_Q84wlkR8rYigoVYsyIPJJ7IkXWdbeJ85YMeAO2DAEKdFSUH32abs7H3CsWuAzYl42Oj02-STPqhtIdV9xdC5CA7EZE-LMCN9KnIxjooBxyuOE1iky1PdWZkNCBUcdWpdV47nX419JF9A-M2gSKORv9_0G_ffmIMTIq9nMgS?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/-qr41TWXFV0pNHoO4VgFf1wDo75iIYlHpStchf9y2yKv-L6n9I30sPZ_dk0bjXBSyDd4XqJ2F5u4vZzZNLO2lVWDwMcGsiN0q8YhwzkCUcG8kENfi38LVA8TeYC1dXqSBBDNv825Qyy9UAVvO6jr96-ewM6EB_Atg_ifBYavouQe6y4Fx2OIJglKye2sx870?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/FvPkm_UbnrkY5VznxQtSum3X6YAh34AywKy5bVMaf1c4bHExNhwu1AwU4Y9DShVqWXn4z3zJbgJcpFswKiTox8Tgs9ZDifl279t4GnUnupRGLEJP5m5lIkl1FCZlxX7OLecyJeY9_TnzDzrofskTgDRz7u3u-zkE2WtchOVdo1zCP8V4Wrb7O07aRyBKC_EJ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/NAUF-UDPczpy7ATatK344LqxPKcztFIU6gPBeuhDsc0UuJvcSDz45e1Q6waSXDfK612sOmZ8C5zbjghzWchdywwpavOBVBM85fhswi-ZlW51orunXMcEL6f9sCpqTAJA0NYOVv3qvLKKtqCpgbUJnRkBMPpVhY0bF7SyXmkOv-cTOZNLl3w6QxR_idaaGZah?purpose=fullsize)

---

# 6. 🐍 `ImageFolder`

PyTorch:

```python
from torchvision.datasets import ImageFolder
```

Example:

```python
train_dataset = ImageFolder(
    root="data/train",
    transform=train_transform
)
```

If the structure is:

```text
train/
├── cat/
└── dog/
```

`ImageFolder` discovers those directories as classes.

Conceptually:

```text
cat → 0
dog → 1
```

and returns:

```python
(image, label)
```

The official torchvision documentation defines `ImageFolder` specifically for this directory-based class organization. ([PyTorch Docs][2])

---

# 7. 🏷️ Labels

Suppose:

```text
cat   → 0
dog   → 1
horse → 2
```

Then:

```text
cat image
   ↓
label = 0
```

A CNN might produce:

$$
[2.1,-0.3,4.7]
$$

These are **logits**.

```text
cat      2.1
dog     -0.3
horse    4.7  ← largest
```

Therefore predicted class:

$$
\boxed{\text{horse}}
$$

---

# 8. ⚠️ Class Imbalance

Suppose:

```text
Cat       9,000
Dog         900
Horse       100
```

Total:

$$
10,000
$$

This is heavily imbalanced.

```text
Cat     ████████████████████
Dog     ██
Horse   ▏
```

---

## 🚨 Why is this dangerous?

Imagine the model predicts:

> **cat for every image**

Accuracy:

$$
\frac{9000}{10000}
=
\boxed{90\%}
$$

😬

The model gets 90% accuracy while essentially failing on the minority classes.

Therefore:

$$
\boxed{\text{High accuracy} \not\Rightarrow \text{good class-wise performance}}
$$

For imbalanced classification, inspect metrics such as:

* per-class recall
* precision
* F1
* confusion matrix
* balanced accuracy

---

# 9. 🖼️ Image Preprocessing

Raw images usually need to be converted into a consistent numerical representation.

Typical pipeline:

```text
Raw image
   ↓
Resize / Crop
   ↓
Tensor conversion
   ↓
Normalize
   ↓
CNN
```

PyTorch's transforms API is specifically designed to manipulate incoming data into a form suitable for model training. ([PyTorch Docs][3])

---

# 10. 📐 Resize

Suppose your dataset contains:

```text
Image A → 640 × 480
Image B → 1920 × 1080
Image C → 300 × 300
```

but the CNN expects:

$$
224\times224
$$

You need to create a consistent input size.

```text
640×480
   ↓
resize
   ↓
224×224
```

---

## ⚠️ Aspect Ratio Problem

Suppose:

$$
800\times400
$$

is directly resized to:

$$
224\times224
$$

The aspect ratio changes:

```text
Original                 Distorted

████████████             █████
████████████             █████
```

The object can become stretched.

That's why resize + crop strategies are commonly used.

---

# 11. ✂️ Crop

Cropping selects a spatial region.

```text
Original
┌─────────────────────┐
│                     │
│        🐶           │
│                     │
│                     │
└─────────────────────┘
           ↓
         Crop
           ↓
      ┌─────────┐
      │   🐶    │
      └─────────┘
```

Cropping can:

* produce the desired input dimensions
* avoid some aspect-ratio distortion
* provide different spatial views

---

# 12. 🎲 Random Crop

During training:

```text
Original image
      ↓
random crop
      ↓
resize
      ↓
CNN
```

Every time an image is loaded, a different crop can potentially be selected.

This makes the crop an **augmentation**, not merely deterministic preprocessing.

---

# 13. 🌈 Color Transformations

Possible transformations include:

* brightness
* contrast
* saturation
* hue

Conceptually:

```text
Same object
     │
 ┌───┼──────────┐
 ↓   ↓          ↓
bright  darker  different saturation
 │   │          │
 └───┼──────────┘
     ↓
same label
```

The purpose is to help the model become less dependent on irrelevant lighting/color variation.

---

# 14. 🔄 Tensor Conversion

Images often begin as PIL images or other image representations.

The model expects tensors.

For an RGB image:

$$
\boxed{C\times H\times W}
$$

Example:

$$
3\times224\times224
$$

For a batch:

$$
\boxed{B\times C\times H\times W}
$$

Example:

$$
32\times3\times224\times224
$$

So:

```text
32 images
   ↓
[32, 3, 224, 224]
```

PyTorch's current transforms-v2 documentation uses `ToImage()` followed by `ToDtype(..., scale=True)` for converting image data into appropriately scaled tensors; older/common pipelines use `ToTensor()`. ([PyTorch Docs][3])

---

# 15. 📊 Normalize

Normalization commonly uses:

$$
\boxed{x'=\frac{x-\mu}{\sigma}}
$$

where:

* \(x\) = input value
* \(\mu\) = mean
* \(\sigma\) = standard deviation

---

## Numerical Example

Suppose:

$$
x=0.8
$$

$$
\mu=0.5
$$

$$
\sigma=0.25
$$

Then:

$$
x'=\frac{0.8-0.5}{0.25}
$$

$$
=\boxed{1.2}
$$

---

# 16. 🧠 Why Normalize?

Normalization puts input values onto a more suitable numerical scale for optimization.

Instead of:

```text
0 → 255
```

you may first scale to:

```text
0 → 1
```

and then normalize around the chosen mean/std.

PyTorch's tutorials describe normalization as adjusting tensor values toward a standardized distribution; their examples use channel-wise means and standard deviations. ([PyTorch Docs][4])

---

# 17. ⚠️ Important: Normalized Images Look Weird

If you directly visualize a normalized tensor as an ordinary image, it can look strange.

That's normal.

To visualize it correctly:

$$
x = x'\sigma+\mu
$$

This is called **unnormalization**.

PyTorch's CIFAR tutorial explicitly reverses the normalization before displaying training images. ([PyTorch Docs][5])

---

# 18. 🔥 Train vs Validation Transforms

This is one of the most important practical rules.

## Training

```python
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
```

## Validation

```python
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])
```

The official PyTorch transfer-learning tutorial uses essentially this pattern: **augmentation + normalization for training, deterministic resize/crop + normalization for validation**. ([PyTorch Docs][6])

### 🔥 Memory rule

```text
TRAIN
Randomness ✅

VALIDATION / TEST
Random augmentation ❌
```

Why?

Because validation/test should provide a **stable measurement**.

---

# 19. 🔥 Data Augmentation

Data augmentation means:

> **Generate varied training views of existing examples using transformations that preserve the intended label.**

```text
                 Original
                    │
        ┌───────────┼───────────┐
        ↓           ↓           ↓
       Flip        Crop        Color
        ↓           ↓           ↓
      Image       Image       Image
```

---

# 20. ↔️ Horizontal Flip

```text
Original             Flipped

🐕 →                  ← 🐕
```

Useful when left/right orientation doesn't change the label.

Example:

```text
dog facing left → dog
dog facing right → dog
```

---

## ⚠️ But not universally valid

If left/right orientation determines the class:

```text
Class A = left-facing sign
Class B = right-facing sign
```

then:

```text
horizontal flip
```

could change the label.

Therefore:

$$
\boxed{\text{Augmentation must be label-preserving}}
$$

---

# 21. 🔄 Rotation

Small rotations can help when real images naturally contain orientation changes.

```text
Original
   ↓
+10°
   ↓
Training example
```

But extreme rotations can create unrealistic examples.

Therefore:

> **Augmentation strength must match the real data distribution.**

---

# 22. ✂️ Random Crop

Random cropping changes which portion of the image is shown.

```text
Original
┌─────────────────┐
│                 │
│       🐶        │
│                 │
└─────────────────┘

       ↓

Random crop

┌───────────┐
│    🐶     │
└───────────┘
```

This can improve robustness to object position and framing.

---

# 23. 🎨 Color Augmentation

Common transformations:

```text
Brightness
Contrast
Saturation
Hue
```

Goal:

```text
Same object
   ↓
different realistic lighting/color
   ↓
same label
```

This encourages the model to focus on features that remain useful despite reasonable appearance changes.

---

# 24. 🔥 Random Resized Crop

This is particularly important.

`RandomResizedCrop`:

1. chooses a random crop area
2. chooses a random aspect ratio within configured limits
3. crops that region
4. resizes it to the requested output size

PyTorch's current documentation describes exactly this behavior. ([PyTorch Docs][7])

```text
Original
   │
   ▼
Random crop
   │
   ├── random area
   └── random aspect ratio
   │
   ▼
Resize
   │
   ▼
224 × 224
```

This is why it is more powerful than simply doing:

```python
Resize(224)
```

The official PyTorch transfer-learning tutorial uses `RandomResizedCrop(224)` for training. ([PyTorch Docs][6])

---

# 25. 🧠 Why Augmentation Helps

Suppose:

$$
N=1000
$$

original images.

Without augmentation:

```text
Model repeatedly sees
essentially the same examples.
```

With stochastic augmentation:

```text
Image 1
 ├── crop A
 ├── crop B
 ├── flip
 ├── color variation
 └── ...
```

The model learns:

> **What remains important despite realistic variation?**

rather than:

> **What exact pixels appeared in this particular training image?**

---

# 26. ⚠️ More Augmentation ≠ Better

### Too little

```text
augmentation ↓
       ↓
less variation
       ↓
overfitting risk ↑
```

### Too much

```text
augmentation ↑↑
       ↓
unrealistic images
       ↓
training signal becomes worse
```

Therefore:

$$
\boxed{\text{Good augmentation}=\text{realistic, label-preserving variation}}
$$

---

# 27. 🚨 DATA LEAKAGE

One of the most important concepts in the module.

Data leakage occurs when information that should be unavailable during training/development **influences the model or its evaluation**.

The result:

```text
Artificially good validation/test performance
             ↓
Poor real-world generalization
```

---

# 28. 💥 Simple Leakage Example

Suppose:

```text
TRAIN
cat001.jpg
cat002.jpg
cat003.jpg
```

and accidentally:

```text
VALIDATION
cat003.jpg
```

Now the exact same image appears in both.

```text
TRAIN
    │
cat003.jpg
    │
    └───────────────┐
                    ↓
               VALIDATION
```

The model has already seen that exact example during training.

Validation performance is therefore contaminated.

---

# 29. 🧬 Duplicate / Near-Duplicate Leakage

It doesn't have to be literally the same filename.

Suppose:

```text
Original photo
     ↓
cropped version
     ↓
TRAIN

Original photo
     ↓
slightly resized version
     ↓
VALIDATION
```

These are effectively related examples.

A model can exploit shared visual information.

Therefore:

> **Deduplication should be considered before splitting when duplicate/near-duplicate images are possible.**

---

# 30. 🏥 Patient / Subject Leakage

This is a major real-world example.

Suppose:

```text
Patient A
 ├── image 1
 ├── image 2
 ├── image 3
 └── image 4
```

If you randomly split individual images:

```text
TRAIN
Patient A → image 1
Patient A → image 2

VALIDATION
Patient A → image 3
Patient A → image 4
```

the model may learn patient-specific characteristics.

Then validation isn't really measuring:

> performance on a new patient.

A better design may be:

```text
TRAIN
Patient A
Patient B
Patient C

VALIDATION
Patient D

TEST
Patient E
```

This is called a **group-aware split** conceptually.

---

# 31. 🕒 Temporal Leakage

Suppose you're building a system that will operate in the future.

Real-world timeline:

```text
January → February → March → April
```

A sensible split might be:

```text
TRAIN
January–February

VALIDATION
March

TEST
April
```

But if you randomly mix every date:

```text
January ─┐
February ├── TRAIN
March ───┼── VALIDATION
April ───┘
```

future information can leak backward into training.

For time-dependent problems:

$$
\boxed{\text{Split according to time when appropriate}}
$$

---

# 32. 📊 Preprocessing Leakage

Suppose you calculate some statistic from:

```text
Train + Validation + Test
```

and use it as part of a learned preprocessing pipeline.

Then held-out information has influenced the transformation.

The safer conceptual procedure is:

```text
TRAIN
  ↓
learn preprocessing statistics
  ↓
fit transformation
  ↓
apply same transformation
to validation/test
```

General rule:

> **Anything learned from data during development should be learned without using held-out evaluation information.**

---

# 33. 🚨 The Correct Data Pipeline

```text
                     RAW DATA
                        │
                        ▼
              Inspect / clean data
                        │
                        ▼
               Remove duplicates
                        │
                        ▼
                 SPLIT FIRST
                        │
       ┌────────────────┼────────────────┐
       ▼                ▼                ▼
     TRAIN             VAL              TEST
       │                │                │
       ▼                │                │
  Augmentation          │                │
       │                │                │
       └─────────┐      │                │
                 ▼      ▼                ▼
               Preprocessing
                    │
                    ▼
              Tensor conversion
                    │
                    ▼
                 Normalize
                    │
                    ▼
                DataLoader
                    │
                    ▼
                   CNN
```

The critical principle is:

$$
\boxed{\text{Split before learning from the data}}
$$

---

# 34. 🚚 DataLoader

After creating a dataset, you normally feed data in **mini-batches**.

```python
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
```

Then:

```python
for images, labels in train_loader:
    ...
```

The batch might have:

```text
images
[32, 3, 224, 224]

labels
[32]
```

PyTorch's tutorials use `DataLoader` to feed batches of transformed images into models. ([PyTorch Docs][1])

---

# 35. 🔀 Why Shuffle Training Data?

Suppose your dataset is ordered:

```text
cat
cat
cat
cat
...
dog
dog
dog
```

Without shuffling:

```text
Batch 1 → cats
Batch 2 → cats
Batch 3 → cats
...
Batch 20 → dogs
```

Training batches can become highly correlated.

With:

```python
shuffle=True
```

the samples are randomized.

```text
Batch 1 → cat dog cat horse
Batch 2 → dog cat horse dog
...
```

For evaluation:

```python
shuffle=False
```

is typically sufficient because you're not learning from those batches. PyTorch's training tutorials demonstrate this distinction. ([PyTorch Docs][8])

---

# 36. 🧪 Complete PyTorch Pipeline

A classic torchvision-style pipeline:

```python
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    "data/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "data/val",
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)
```

This follows the same fundamental pattern used in PyTorch's official transfer-learning tutorial. ([PyTorch Docs][6])

---

# 37. 🔍 Code → Concept Mapping

| Code                   | What it means                       |
| ---------------------- | ----------------------------------- |
| `ImageFolder(...)`     | Load folder-organized image dataset |
| `RandomResizedCrop`    | Random spatial augmentation         |
| `RandomHorizontalFlip` | Random horizontal augmentation      |
| `ToTensor()`           | Convert image to tensor             |
| `Normalize()`          | Normalize channels                  |
| `DataLoader(...)`      | Create mini-batches                 |
| `shuffle=True`         | Randomize training order            |
| `shuffle=False`        | Stable evaluation ordering          |

---

# 38. 📐 Shape Flow

For an RGB image:

```text
Image
   ↓
3 × H × W
```

After resizing:

```text
3 × 224 × 224
```

After batching:

```text
B × 3 × 224 × 224
```

For:

$$
B=32
$$

we get:

$$
\boxed{32\times3\times224\times224}
$$

Then:

```text
[32, 3, 224, 224]
        ↓
       CNN
        ↓
[32, num_classes]
```

For example, 10 classes:

$$
[32,10]
$$

---

# 39. 🧠 Common Mistakes

### ❌ 1. Test-set tuning

Using test accuracy repeatedly to select hyperparameters.

### ❌ 2. Random image splitting when sources are related

Example:

```text
same patient
same video
same object
same subject
```

appearing across splits.

### ❌ 3. Augmenting validation randomly

Makes evaluation unnecessarily noisy.

### ❌ 4. Ignoring class imbalance

Can make accuracy misleading.

### ❌ 5. Excessive augmentation

Can create unrealistic training examples.

### ❌ 6. Wrong normalization

Using inappropriate statistics or applying normalization inconsistently.

### ❌ 7. Resizing without considering aspect ratio

Can distort objects.

### ❌ 8. Forgetting the batch dimension

Single image:

$$
[3,224,224]
$$

Batch:

$$
[32,3,224,224]
$$

### ❌ 9. Assuming `ImageFolder` solves dataset quality

`ImageFolder` handles **organization/loading**.

It doesn't automatically guarantee:

* correct labels
* balanced classes
* absence of duplicates
* leakage-free splitting
* good augmentation

---

# 40. 🎤 Interview Questions

### Q1. What are train, validation and test sets?

> Train learns parameters; validation supports development decisions; test provides final evaluation on held-out data.

---

### Q2. What does `ImageFolder` do?

> It maps subdirectories under a root directory to class labels and loads images accordingly.

---

### Q3. Why normalize images?

> To place input values on a suitable numerical scale for neural-network optimization.

---

### Q4. Why use augmentation only during training?

> To increase realistic training variation without making validation/test evaluation stochastic.

---

### Q5. What is data leakage?

> When information that should be unavailable during training or model development influences the model or its evaluation.

---

### Q6. Give a subtle leakage example.

> Multiple images from the same patient, video, or physical subject are split across training and validation sets.

---

### Q7. Why is class imbalance dangerous?

> A model can achieve high overall accuracy by predicting the majority class while performing poorly on minority classes.

---

### Q8. What does `RandomResizedCrop` do?

> It randomly selects a crop with a random area/aspect ratio within configured ranges and resizes the crop to the requested output size. ([PyTorch Docs][7])

---

# 41. 🧠 Ultimate Mental Model

Don't memorize:

```text
Resize
Crop
Flip
Normalize
DataLoader
```

as disconnected PyTorch APIs.

Understand the **purpose** of each:

```text
RAW IMAGE
   │
   ▼
"How should my dataset be organized?"
   │
   ▼
ImageFolder
   │
   ▼
"How do I create a fair evaluation?"
   │
   ▼
Train / Val / Test
   │
   ▼
"Are classes balanced?"
   │
   ▼
Class-distribution check
   │
   ▼
"How can I make training examples varied?"
   │
   ▼
Augmentation
   │
   ▼
"How do I give the CNN consistent input?"
   │
   ▼
Resize / Crop
   │
   ▼
"How do I convert it into model input?"
   │
   ▼
Tensor
   │
   ▼
"How do I put values on a useful scale?"
   │
   ▼
Normalize
   │
   ▼
"How do I feed efficient mini-batches?"
   │
   ▼
DataLoader
   │
   ▼
CNN
```

---

# 🔥 THE ONE THING TO REMEMBER

The CNN isn't the whole system.

```text
             MODEL
               ▲
               │
          DataLoader
               ▲
               │
        Preprocessing
               ▲
               │
         Augmentation
               ▲
               │
       Train/Val/Test
               ▲
               │
            Dataset
```

If the **dataset split is contaminated**, the evaluation is wrong.

If the **labels are wrong**, the model learns the wrong mapping.

If the **preprocessing is wrong**, optimization suffers.

If the **augmentation is unrealistic**, the training signal becomes harmful.

If the **classes are severely imbalanced**, accuracy may lie.

So:

$$
\boxed{
\text{Good CNN}
+
\text{Good Data Pipeline}
=
\text{Meaningful Training}
}
$$

---

# ⚡ 30-SECOND REVISION

> **A CNN image-training pipeline begins with a properly organized dataset and leakage-free train/validation/test split. `ImageFolder` can map class folders to labels, while transforms handle resizing, cropping, tensor conversion and normalization. Training data can use label-preserving augmentation such as horizontal flips, random crops, color changes and random resized crops, whereas validation/test preprocessing should remain deterministic. `DataLoader` converts the dataset into mini-batches. Class imbalance and data leakage must be checked because they can make evaluation misleading.**

---

# 📌 MASTER CHEAT SHEET

| Concept           | Remember                                                    |
| ----------------- | ----------------------------------------------------------- |
| Dataset sample    | `(image, label)`                                            |
| Train             | Learn weights                                               |
| Validation        | Development decisions                                       |
| Test              | Final held-out evaluation                                   |
| `ImageFolder`     | Folder → class mapping                                      |
| Resize            | Standardize spatial size                                    |
| Crop              | Select spatial region                                       |
| Tensor conversion | Image → model tensor                                        |
| RGB shape         | `C×H×W`                                                     |
| Batch shape       | `B×C×H×W`                                                   |
| Normalize         | `(x−μ)/σ`                                                   |
| Flip              | Spatial augmentation                                        |
| Rotation          | Orientation variation                                       |
| Color transform   | Lighting/color variation                                    |
| RandomResizedCrop | Random crop + resize                                        |
| Augmentation      | Increase realistic training variation                       |
| Class imbalance   | Unequal class frequencies                                   |
| Leakage           | Held-out information contaminates development/evaluation    |
| Train shuffle     | Usually `True`                                              |
| Val/test shuffle  | Usually `False`                                             |
| Core rule         | **Split carefully, augment training, stabilize evaluation** |

---